# Week 16, Day 1: Divide and conquer

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w016d1.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
n, steps = 1000, 0
while n > 1:
    n = (n + 1) // 2
    steps += 1
print(steps)

In [ ]:
def largest(items):
    if len(items) == 1:
        return items[0]
    mid = len(items) // 2
    left = largest(items[:mid])
    right = largest(items[mid:])
    return left if left > right else right

print(largest([3, 9, 4, 1, 7, 2, 8]))

*Interactive figure on the website: Each call splits its list in two and waits for both halves. The tree is only about log2(n) levels deep, however long the list..*

> **Key idea.** Divide and conquer: split the problem (usually in half), solve each part recursively, and combine the answers. The hard part, and the part to think about first, is usually the **combine** step.

In [ ]:
def search(items, target, lo=0, hi=None):
    """The position of target in the sorted list, or -1."""
    if hi is None:
        hi = len(items) - 1
    if lo > hi:
        return -1
    mid = (lo + hi) // 2
    if items[mid] == target:
        return mid
    if target < items[mid]:
        return search(items, target, lo, mid - 1)
    return search(items, target, mid + 1, hi)

primes = [2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47]
print(search(primes, 31), search(primes, 2), search(primes, 47), search(primes, 20))

> **Common mistake.** Binary search only works on **sorted** data. On an unsorted list it quietly gives wrong answers, because "the target must be in the left half" is no longer true. You'll study it, and its famously tricky off-by-one details, in Week 18.

In [ ]:
def merge(a, b):
    result, i, j = [], 0, 0
    while i < len(a) and j < len(b):
        if a[i] <= b[j]:
            result.append(a[i])
            i += 1
        else:
            result.append(b[j])
            j += 1
    return result + a[i:] + b[j:]

def merge_sort(items):
    if len(items) <= 1:
        return list(items)
    mid = len(items) // 2
    return merge(merge_sort(items[:mid]), merge_sort(items[mid:]))

print(merge_sort([38, 27, 43, 3, 9, 82, 10]))

In [ ]:
import math
for n in [8, 1000, 10 ** 6, 10 ** 9]:
    print(f"{n:>13,}  {math.ceil(math.log2(n))} levels")

### Exercise: Sum by halves

Write `dc_sum(items)` that adds up a list by divide and conquer: split in half, add up each half recursively, and add the two results. The empty list sums to 0.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert dc_sum([3, 9, 4, 1, 7]) == 24 and dc_sum([]) == 0 and dc_sum([5]) == 5\n    assert dc_sum(list(range(10000))) == sum(range(10000))\n    import re\n    assert not re.search(r\"\\bsum\\(\", source) and \"for \" not in source and \"while \" not in source")

### Exercise: Count by halves

Write `dc_count(items, target)` that counts how many times `target` appears, by divide and conquer, passing bounds `lo` and `hi` instead of slicing.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert dc_count([1, 2, 1, 1, 3], 1) == 3 and dc_count([], 1) == 0 and dc_count([2], 2) == 1 and dc_count([2], 3) == 0\n    assert dc_count(list(range(5000)) * 2, 7) == 2\n    assert \".count(\" not in source and \"for \" not in source and \"while \" not in source")

### Exercise: First position

Write `first_at_least(items, target)` for a **sorted** list: the position of the first item that's greater than or equal to `target`, or `len(items)` if there isn't one. Use recursive binary search with bounds. (This is where `target` would be inserted to keep the list sorted.)

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    v = [1, 3, 3, 3, 7, 9]\n    assert [first_at_least(v, t) for t in (0, 1, 2, 3, 4, 9, 10)] == [0, 0, 1, 1, 4, 5, 6]\n    assert first_at_least([], 5) == 0\n    import bisect\n    big = list(range(0, 100000, 3))\n    assert all(first_at_least(big, t) == bisect.bisect_left(big, t) for t in range(-2, 100002, 997))\n    assert \"bisect\" not in source and \"for \" not in source and \"while \" not in source")

### Exercise: Merge sort, by key

Write `merge_sort(items, key)` that sorts a list using merge sort, comparing `key(item)` instead of the items themselves, and keeping items with equal keys in their original order (a **stable** sort). Use `<=` in the merge so that, on a tie, the item from the left half comes first. Don't use `sort` or `sorted`.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    words = [\"pear\", \"Fig\", \"apple\", \"kiwi\", \"date\", \"banana\"]\n    assert merge_sort(words, len) == [\"Fig\", \"pear\", \"kiwi\", \"date\", \"apple\", \"banana\"]\n    assert merge_sort([3, 1, 2], lambda x: -x) == [3, 2, 1]\n    assert merge_sort([], len) == []\n    import random\n    r = random.Random(4)\n    data = [(r.randint(0, 9), i) for i in range(500)]\n    assert merge_sort(data, lambda p: p[0]) == sorted(data, key=lambda p: p[0])\n    assert \"sort(\" not in source.replace(\"merge_sort(\", \"\") and \"sorted(\" not in source")

### Exercise: Both at once

Write `min_max(items)` that returns a tuple `(smallest, largest)` of a non-empty list by divide and conquer, combining the two halves' tuples. Raise a `ValueError` for an empty list.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert min_max([3, 9, 4, 1, 7]) == (1, 9) and min_max([5]) == (5, 5) and min_max([2, 1]) == (1, 2)\n    try:\n        min_max([])\n    except ValueError:\n        pass\n    else:\n        raise AssertionError(\"an empty list should raise a ValueError\")\n    import re\n    assert not re.search(r\"\\b(min|max)\\(\", source), \"don't use min or max\"")

### Bug hunt: It never ends

This binary search goes into infinite recursion (a `RecursionError`) when the target isn't in the list, and sometimes even when it is. Fix the two recursive calls.

In [ ]:
def search(items, target, lo=0, hi=None):
    if hi is None:
        hi = len(items) - 1
    if lo > hi:
        return -1
    mid = (lo + hi) // 2
    if items[mid] == target:
        return mid
    if target < items[mid]:
        return search(items, target, lo, mid)
    return search(items, target, mid, hi)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"found and not found\"\n    v = [2, 4, 6, 8, 10]\n    assert [search(v, t) for t in (2, 4, 6, 8, 10)] == [0, 1, 2, 3, 4]\n    assert search(v, 5) == -1 and search(v, 11) == -1 and search([], 1) == -1")

### Puzzle: Merge sort

Arrange the lines for `merge_sort`, given a working `merge`. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them

Work these out without running them.

In [ ]:
a = ___   # the most calls binary search makes on a sorted list of 1,000 items (base case included): 10, 11, 500 or 1000?
b = ___   # merge([1, 5, 9], [2, 3, 10])
c = ___   # how many levels of calls deep does merge_sort go for a list of 16 items, counting the first call as level 1?
d = ___   # search([1, 3, 5, 7], 7): how many calls, counting the first?

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert a == 11\n    assert b == [1, 2, 3, 5, 9, 10]\n    assert c == 5\n    assert d == 3")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: the best stretch

Given daily profits and losses, what's the best total you could have made over any stretch of **consecutive** days (at least one day)? For `[2, -5, 6, -2, 3, -9, 4]` it's 7 (days 2 to 4: 6 - 2 + 3). Solve it by divide and conquer: the best stretch is entirely in the left half, entirely in the right half, or it **crosses the middle**. The first two are recursive calls. For the crossing one, find the best total ending exactly at the middle (going left from it) and the best total starting just after it (going right), and add them. Write `best_stretch(values)`, raising a `ValueError` for an empty list.

(There's also a famous one-pass method, Kadane's algorithm, which you could write as a check; but the point here is the divide and conquer.)

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def brute(v):\n    return max(sum(v[i:j]) for i in range(len(v)) for j in range(i + 1, len(v) + 1))\ndef test_values():\n    \"several\"\n    assert best_stretch([2, -5, 6, -2, 3, -9, 4]) == 7\n    assert best_stretch([-3, -1, -7]) == -1 and best_stretch([5]) == 5 and best_stretch([1, 2, 3]) == 6\n    import random\n    r = random.Random(7)\n    for _ in range(100):\n        v = [r.randint(-20, 20) for _ in range(r.randint(1, 30))]\n        assert best_stretch(v) == brute(v), v\n    try:\n        best_stretch([])\n    except ValueError:\n        pass\n    else:\n        raise AssertionError(\"an empty list should raise a ValueError\")")

## Warm-up

Think of a number from 1 to 1,000. If I may ask "is it bigger than...?", how many questions do I need, at most? Not 1,000, and not 500: **10**. Each question halves the numbers still possible: 1,000, 500, 250, 125, 63, 32, 16, 8, 4, 2, 1. Halving shrinks problems astonishingly fast, and that's the power behind today's idea.

## Divide, conquer, combine

**Divide and conquer** solves a problem in three steps:

1. **Divide** it into smaller problems of the same kind, usually two halves.
2. **Conquer** each smaller problem recursively (the leap of faith again). Very small problems are solved directly: the base case.
3. **Combine** the answers to the smaller problems into the answer for the whole.

Here's the largest item of a list, divided and conquered:

The combine step is easy here: the largest of the whole list is the larger of the two halves' largests. Step through it and see the tree of calls, two branches at each level:

For finding the largest, a loop is simpler and just as fast. Divide and conquer earns its keep when the halves make the problem **easier**, as in the next two sections.

## Binary search

To find a value in a **sorted** list, look at the middle item. If it's the one, done. If the target is smaller, it can only be in the left half; if larger, only in the right half. Search that half the same way. This is **binary search**, the number game from the warm-up, on a list.

Copying half the list each time would waste the speed, so instead pass the **bounds** of the part still to search: `lo` and `hi`, the first and last positions it could be in.

The base case is `lo > hi`: the part still to search is empty, so the target isn't there. Each call halves the part left, so a million sorted items take at most 20 calls. Here, there's no combine step at all: only one half needs searching.

## Merge sort

Here's divide and conquer at its most powerful. To sort a list:

1. **Divide** it into two halves.
2. **Conquer**: sort each half (recursively).
3. **Combine**: **merge** the two sorted halves into one sorted list, which you did in the Phase 1 exam: repeatedly take the smaller of the two front items.

A list of 0 or 1 items is already sorted: the base case.

This is **merge sort**, invented by John von Neumann in 1945. It sorts a million items in about 20 million steps, where the simple sorts you'll meet in Week 18 need around a million million. Week 19 looks at exactly why; for now, notice the shape. The hard work happens in the **combine** step, and it's easy only because the halves are already sorted.

## How deep does it go?

Halving is what keeps divide and conquer practical. Each level of calls halves the size, so the depth of the recursion is about **log2(n)**: the number of times you can halve n before reaching 1.

| n | levels (about log2 n) |
|---|---|
| 8 | 3 |
| 1,000 | 10 |
| 1,000,000 | 20 |
| 1,000,000,000 | 30 |

So divide and conquer never gets anywhere near Python's recursion limit: a billion items need only 30 levels. Compare last week's `items[0] + total(items[1:])`, which needed one level per item.

## Practice

## Quiz

## Daily challenge

## Before you go

For each of today's problems, name the three steps: how it divides, what the recursive calls solve, and how it combines. Then think of a real task you do by halving (finding a page in a book, a word in a dictionary, a bad step in a recipe that went wrong). Tomorrow: tree recursion, where a function makes several recursive calls and explores every possibility.